In [1]:
import pandas as pd
pd.set_option('display.max_columns', 200)

PROJECT_DIR = "/Users/hoyin/Desktop/DanaFarber/workspaces/CMI_Painter_Angiosarcoma_WES_analysis_mh_regional/scripts"
RAW_DATA_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DATA_DIR = f"{PROJECT_DIR}/data/processed"


In [2]:
## Load the Terra tables (and other sample related characteristics tables)
dna_pair_table = pd.read_csv(f"{RAW_DATA_DIR}/sample_tables/pair_Jan13_2024.tsv",sep="\t")
dna_sample_table = pd.read_csv(f"{RAW_DATA_DIR}/sample_tables/sample_Jan13_2024.tsv",sep="\t")
rna_sample_table = pd.read_csv(f"{RAW_DATA_DIR}/sample_tables/RNA_sample_Feb19_2024.tsv",sep="\t")


In [3]:
## Load the timepoint df that has sample-level informaton
timepoint_df = pd.read_excel(f"{RAW_DATA_DIR}/ASC Timepoints data.xlsx",skiprows=2)
timepoint_df["sample_alias"] = timepoint_df["SAMPLE_TIMEPOINT"].str.replace("__","_",regex=True)
timepoint_df["sample_alias"] = timepoint_df["SAMPLE_TIMEPOINT"].str.replace("_ ","_",regex=True)


## Clean up some of the original naming scheme
bx_location_dict = {
    "Right breast": "BREAST",
    "right breast":"BREAST",
    "BREAST,AXILLA":"BREAST",
    "Skin, Left breast": "BREAST",
    "Skin, right chest": "CHEST",
    "Skin and subcutaneous tissue, chest":"CHEST",
    "CHEST SKIN": "CHEST",
    "CHEST WALL": "CHEST",
    "Liver":"LIVER",
    "Right Kidney": "KIDNEY",
    "Leg":"BONELIMB",
    "BONE":"BONELIMB",
    "LEG":"BONELIMB",
    "Right medial 4th, 5th, 6th ribs":"BONELIMB",
    "SHOULDER":"BONELIMB",
    "Scalp":"HEADFACENECKSCALP",
    "Skin, Right Nasal": "HEADFACENECKSCALP",
    "SCALP":'HEADFACENECKSCALP',
    "FACE": "HEADFACENECKSCALP",
    "TEMPLE":"HEADFACENECKSCALP",
    "PERICARDIAL FLUID":"HEART",
    "RIGHT ATRIUM":"HEART"
    
}
timepoint_df["BX_LOCATION_OG"] = timepoint_df["BX_LOCATION"]
timepoint_df["BX_LOCATION"] = timepoint_df["BX_LOCATION"].replace(bx_location_dict)

## Clean up BX type entries
timepoint_df["BX_VASOFORMATIVE"] = timepoint_df["BX_VASOFORMATIVE"].str.upper().replace({
    "YES; FOCAL": "YES (FOCAL)","NO ":"NO"
})
timepoint_df["BX_EPITHELIOID"] = timepoint_df["BX_EPITHELIOID"].str.upper().replace({
    "YES; FOCAL":"YES (FOCAL)","NO ":"NO"
})
timepoint_df["BX_SPINDLE_CELL"] = timepoint_df["BX_SPINDLE_CELL"].str.upper().replace({
    "YES; FOCAL":"YES (FOCAL)","NO ":"NO","YES ":"YES"
})
timepoint_df["BX_NUCLEAR_GRADE"] = timepoint_df["BX_NUCLEAR_GRADE"].str.upper()
timepoint_df["BX_IS_BLOOD_BIOPSY"] = timepoint_df["SAMPLE_TIMEPOINT"].str.contains("BLOOD")
timepoint_df


,PATIENT_ID,SAMPLE_TIMEPOINT,CANCER_TYPE,CANCER_TYPE_DETAILED,BX_TIME_DAYS,BX_LOCATION,BX_TYPE,BX_VASOFORMATIVE,BX_EPITHELIOID,BX_SPINDLE_CELL,BX_NUCLEAR_GRADE,sample_alias,BX_LOCATION_OG,BX_IS_BLOOD_BIOPSY
0,ASCProject_0007,ASCProject_0007_T1,Soft Tissue Sarcoma,Angiosarcoma,181.0,BLADDER WALL,RESECTION,YES,YES,YES,HIGH,ASCProject_0007_T1,BLADDER WALL,False
1,ASCProject_0222,ASCProject_0222_T2,Soft Tissue Sarcoma,Angiosarcoma,28.0,HEADFACENECKSCALP,EXCISION,YES,YES,NO,HIGH,ASCProject_0222_T2,HEADFACENECKSCALP,False
2,ASCProject_0290,ASCProject_0290_T1,Soft Tissue Sarcoma,Angiosarcoma,0.0,SPLEEN,SPLENECTOMY,YES (FOCAL),NO,YES,HIGH,ASCProject_0290_T1,SPLEEN,False
3,ASCProject_0135,ASCProject_0135_T2,Soft Tissue Sarcoma,Angiosarcoma,37.0,BREAST,MASTECTOMY,NO,YES,NO,HIGH,ASCProject_0135_T2,BREAST,False
4,ASCProject_0045,ASCProject_0045_T1,Soft Tissue Sarcoma,Angiosarcoma,14.0,BREAST,MASTECTOMY,YES,NO,YES,HIGH,ASCProject_0045_T1,BREAST,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,ASCProject_0179,ASCProject_0179_T1,Soft Tissue Sarcoma,Angiosarcoma,33.0,BREAST,NaN,NaN,NaN,NaN,NaN,ASCProject_0179_T1,BREAST,False
117,ASCProject_0194,ASCProject_0194__T1,Soft Tissue Sarcoma,Angiosarcoma,-1.0,HEART,NaN,NaN,NaN,NaN,NaN,ASCProject_0194__T1,PERICARDIAL FLUID,False
118,ASCProject_0212,ASCProject_0212_T1,Soft Tissue Sarcoma,Angiosarcoma,0.0,HEART,NaN,NaN,NaN,NaN,NaN,ASCProject_0212_T1,HEART,False
119,ASCProject_0277,ASCProject_0277_T1,Soft Tissue Sarcoma,Angiosarcoma,0.0,HEADFACENECKSCALP,NaN,NaN,NaN,NaN,NaN,ASCProject_0277_T1,FACE,False


In [4]:
## Process DNA Table
dna_table_subset = dna_sample_table[["entity:sample_id","sample_alias","sample_type","individual_alias"]].copy()
dna_table_subset["entity:sample_id"] = dna_table_subset["entity:sample_id"].replace("__","_",regex=True)
dna_table_subset["entity:sample_id"] = dna_table_subset["entity:sample_id"].replace("_ ","_",regex=True)
dna_table_subset["sample_alias"] = dna_table_subset["sample_alias"].replace("__","_",regex=True)
dna_table_subset["sample_alias"] = dna_table_subset["sample_alias"].replace("_ ","_",regex=True)
dna_table_subset["data_type"] = "DNA"

dna_table_subset["tumor_timepoint"] = dna_table_subset["entity:sample_id"].apply(lambda x:x.split("_")[3].strip())
dna_table_subset.loc[dna_table_subset["sample_type"]!="Tumor","tumor_timepoint"] = None
dna_table_subset["sample_alias_cleaned"] = dna_table_subset["sample_alias"]
dna_table_subset

,entity:sample_id,sample_alias,sample_type,individual_alias,data_type,tumor_timepoint,sample_alias_cleaned
0,RP-1447_ASCProject_0003_SALIVA_v1_Exome_OnPrem,ASCProject_0003_SALIVA,Normal,ASCProject_0003,DNA,None,ASCProject_0003_SALIVA
1,RP-1447_ASCProject_0003_T1_v1_Exome_OnPrem,ASCProject_0003_T1,Tumor,ASCProject_0003,DNA,T1,ASCProject_0003_T1
2,RP-1447_ASCProject_0003_T3_v2_Exome_OnPrem,ASCProject_0003_T3,Tumor,ASCProject_0003,DNA,T3,ASCProject_0003_T3
3,RP-1447_ASCProject_0005_BLOOD_Normal_v1_Exome_...,ASCProject_0005_BLOOD_Normal,Normal,ASCProject_0005,DNA,None,ASCProject_0005_BLOOD_Normal
4,RP-1447_ASCProject_0005_BLOOD_v1_Exome_OnPrem,ASCProject_0005_BLOOD,Tumor,ASCProject_0005,DNA,BLOOD,ASCProject_0005_BLOOD
...,...,...,...,...,...,...,...
365,RP-1447_ASCProject_PTLPDP_SALIVA_v1_Exome_OnPrem,ASCProject_PTLPDP_SALIVA,Normal,ASCProject_PTLPDP,DNA,None,ASCProject_PTLPDP_SALIVA
366,RP-1447_ASCProject_PY6FJH_SALIVA_v1_Exome_OnPrem,ASCProject_PY6FJH_SALIVA,Normal,ASCProject_PY6FJH,DNA,None,ASCProject_PY6FJH_SALIVA
367,RP-1447_ASCProject_PYMFBV_SALIVA_v1_Exome_OnPrem,ASCProject_PYMFBV_SALIVA,Normal,ASCProject_PYMFBV,DNA,None,ASCProject_PYMFBV_SALIVA
368,RP-1447_ASCProject_PYYKC9_SALIVA_v1_Exome_OnPrem,ASCProject_PYYKC9_SALIVA,Normal,ASCProject_PYYKC9,DNA,None,ASCProject_PYYKC9_SALIVA


In [5]:
## Process RNA Table
rna_table_subset = rna_sample_table[[
    "entity:sample_id","LC-SET","sample_alias","sample_type","individual_alias"
]].copy()
rna_table_subset["LC_BATCH"] = rna_table_subset["LC-SET"].apply(lambda x: x.split(",")[0])
rna_table_subset["data_type"] = "RNA"

# Clean name formatting
rna_table_subset["sample_alias_cleaned"] = rna_table_subset["sample_alias"].str.replace(
    "_UNKNOWN_","_"
).str.replace("_Whitelisted","")

rna_table_subset["tumor_timepoint"] = rna_table_subset["sample_alias_cleaned"].apply(lambda x:x.split("_")[-1].strip())
rna_table_subset.loc[rna_table_subset["sample_type"]!="Tumor","tumor_timepoint"] = None
rna_table_subset


,entity:sample_id,LC-SET,sample_alias,sample_type,individual_alias,LC_BATCH,data_type,sample_alias_cleaned,tumor_timepoint
0,G136942_ASCProject_0005_T1_v1_RNA_OnPrem,LCSET-12154,ASCProject_0005_T1,Tumor,ASCProject_0005,LCSET-12154,RNA,ASCProject_0005_T1,T1
1,G136942_ASCProject_0007_T1_v1_RNA_OnPrem,LCSET-12154,ASCProject_0007_T1,Tumor,ASCProject_0007,LCSET-12154,RNA,ASCProject_0007_T1,T1
2,G136942_ASCProject_0007_T2_v1_RNA_OnPrem,LCSET-12154,ASCProject_0007_T2,Tumor,ASCProject_0007,LCSET-12154,RNA,ASCProject_0007_T2,T2
3,G136942_ASCProject_0026_T1_v1_RNA_OnPrem,LCSET-12154,ASCProject_0026_T1,Tumor,ASCProject_0026,LCSET-12154,RNA,ASCProject_0026_T1,T1
4,G136942_ASCProject_0042_T1_v1_RNA_OnPrem,LCSET-12154,ASCProject_0042_T1,Tumor,ASCProject_0042,LCSET-12154,RNA,ASCProject_0042_T1,T1
...,...,...,...,...,...,...,...,...,...
154,RP-1447_ASCProject_PHCG7A_T1_v1_RNA_OnPrem,LCSET-26033,ASCProject_PHCG7A_T1,Tumor,ASCProject_PHCG7A,LCSET-26033,RNA,ASCProject_PHCG7A_T1,T1
155,RP-1447_ASCProject_PRDZ3M_T2_v1_RNA_OnPrem,LCSET-20590,ASCProject_PRDZ3M_T2,Tumor,ASCProject_PRDZ3M,LCSET-20590,RNA,ASCProject_PRDZ3M_T2,T2
156,RP-1447_ASCProject_PWNLUF_T1_v1_RNA_OnPrem,LCSET-20590,ASCProject_PWNLUF_T1,Tumor,ASCProject_PWNLUF,LCSET-20590,RNA,ASCProject_PWNLUF_T1,T1
157,RP-1447_ASCProject_PXPTMJ_T1_v1_RNA_OnPrem,LCSET-18352,ASCProject_PXPTMJ_T1,Tumor,ASCProject_PXPTMJ,LCSET-18352,RNA,ASCProject_PXPTMJ_T1,T1


In [6]:
## Keep only the sample information required
timepoint_df_subset = timepoint_df[
    [
        "sample_alias","BX_TIME_DAYS","BX_LOCATION",
        "BX_TYPE","BX_VASOFORMATIVE","BX_EPITHELIOID","BX_SPINDLE_CELL","BX_NUCLEAR_GRADE"
    ]
]

## Combine the tables
integrated_table = pd.concat([dna_table_subset,rna_table_subset])
integrated_table = integrated_table.merge(timepoint_df_subset,how="left",on="sample_alias")
integrated_table

,entity:sample_id,sample_alias,sample_type,individual_alias,data_type,tumor_timepoint,sample_alias_cleaned,LC-SET,LC_BATCH,BX_TIME_DAYS,BX_LOCATION,BX_TYPE,BX_VASOFORMATIVE,BX_EPITHELIOID,BX_SPINDLE_CELL,BX_NUCLEAR_GRADE
0,RP-1447_ASCProject_0003_SALIVA_v1_Exome_OnPrem,ASCProject_0003_SALIVA,Normal,ASCProject_0003,DNA,None,ASCProject_0003_SALIVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RP-1447_ASCProject_0003_T1_v1_Exome_OnPrem,ASCProject_0003_T1,Tumor,ASCProject_0003,DNA,T1,ASCProject_0003_T1,NaN,NaN,13.0,BREAST,RESECTION,YES,NO,NO,LOW
2,RP-1447_ASCProject_0003_T3_v2_Exome_OnPrem,ASCProject_0003_T3,Tumor,ASCProject_0003,DNA,T3,ASCProject_0003_T3,NaN,NaN,26.0,BREAST,NaN,NaN,NaN,NaN,NaN
3,RP-1447_ASCProject_0005_BLOOD_Normal_v1_Exome_...,ASCProject_0005_BLOOD_Normal,Normal,ASCProject_0005,DNA,None,ASCProject_0005_BLOOD_Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RP-1447_ASCProject_0005_BLOOD_v1_Exome_OnPrem,ASCProject_0005_BLOOD,Tumor,ASCProject_0005,DNA,BLOOD,ASCProject_0005_BLOOD,NaN,NaN,4338.0,BLOOD BIOPSY,BLOOD BIOPSY,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524,RP-1447_ASCProject_PHCG7A_T1_v1_RNA_OnPrem,ASCProject_PHCG7A_T1,Tumor,ASCProject_PHCG7A,RNA,T1,ASCProject_PHCG7A_T1,LCSET-26033,LCSET-26033,NaN,NaN,NaN,NaN,NaN,NaN,NaN
525,RP-1447_ASCProject_PRDZ3M_T2_v1_RNA_OnPrem,ASCProject_PRDZ3M_T2,Tumor,ASCProject_PRDZ3M,RNA,T2,ASCProject_PRDZ3M_T2,LCSET-20590,LCSET-20590,NaN,NaN,NaN,NaN,NaN,NaN,NaN
526,RP-1447_ASCProject_PWNLUF_T1_v1_RNA_OnPrem,ASCProject_PWNLUF_T1,Tumor,ASCProject_PWNLUF,RNA,T1,ASCProject_PWNLUF_T1,LCSET-20590,LCSET-20590,NaN,NaN,NaN,NaN,NaN,NaN,NaN
527,RP-1447_ASCProject_PXPTMJ_T1_v1_RNA_OnPrem,ASCProject_PXPTMJ_T1,Tumor,ASCProject_PXPTMJ,RNA,T1,ASCProject_PXPTMJ_T1,LCSET-18352,LCSET-18352,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
integrated_table.to_csv(f"{PROJECT_DIR}/data/processed/sample_data.tsv",sep="\t",index=False)

## Merge sample and patient level data

In [8]:
#clin_data = pd.read_csv(f"{PROCESSED_DATA_DIR}/clinical_data.tsv",sep="\t")
clin_data = pd.read_csv(f"{PROCESSED_DATA_DIR}/clinical_data_manual_deident.tsv",sep="\t")

sample_patient_merged = integrated_table.merge(
    clin_data,
    left_on="individual_alias",
    right_on="STUDY ID",
    how="left"
)
sample_patient_merged

,entity:sample_id,sample_alias,sample_type,individual_alias,data_type,tumor_timepoint,sample_alias_cleaned,LC-SET,LC_BATCH,BX_TIME_DAYS,BX_LOCATION,BX_TYPE,BX_VASOFORMATIVE,BX_EPITHELIOID,BX_SPINDLE_CELL,BX_NUCLEAR_GRADE,STUDY ID,Primary Site (Recombined),Mets Sites Ever (Recombined),Primary Site (Wagner2024),Biopsy Location (Wagner2024),Met Sites at Dx (Wagner2024),Met Sites Ever (Wagner2024),Other Cancer,BATCH NUMBER (EHR_EXTRACTED),AGE AT DX (EHR_EXTRACTED),SEX (EHR_EXTRACTED),PRIMARY SITE (EHR_EXTRACTED),DX BIOPSY LOCATION (EHR_EXTRACTED),RAAS (EHR_EXTRACTED),LAAS/STEWART-TREVES (EHR_EXTRACTED),CUTANEOUS AS (EHR_EXTRACTED),LOCAL RECURRENCE (EHR_EXTRACTED),LOCAL RECURRENCE SITE (EHR_EXTRACTED),LOCAL RECURRENCE DATE 1 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 2 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 3 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 4 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 5 (DAYS FROM DX) (EHR_EXTRACTED),LOCAL RECURRENCE DATE 6 (DAYS FROM DX) (EHR_EXTRACTED),DATE METS DX (DAYS FROM DX) (EHR_EXTRACTED),METS SITES AT METS DX (EHR_EXTRACTED),METS SITES EVER (EHR_EXTRACTED),FINAL DATE MEDR (DAYS FROM DX) (EHR_EXTRACTED),PRIMARY SITE STANDARDIZED (EHR_EXTRACTED),LAAS (EHR_EXTRACTED),HAS METS AT DX (EHR_EXTRACTED),legacy_shortid (PRD),PREQUAL_SELF_DESCRIBE (PRD),AGE OF DIAGNOSIS (PRD),DIAGNOSIS_LOC (PRD),EVER_LOCATION (PRD),CURRENT_LOCATION (PRD),SURGERY (PRD),SURGERY_CLEAN_MARGINS (PRD),RADIATION (PRD),RADIATION_SURGERY (PRD),CURRENTLY_TREATED (PRD),OTHER_CANCER (PRD),OTHER_CANCER_RADIATION (PRD),HISPANIC (PRD),RACE (PRD),DIAGNOSIS_PRIMARY_LOC (PRD),DIAGNOSIS_SPREAD (PRD),DIAGNOSIS_SPREAD_LOC (PRD),POST_DIAGNOSIS_SPREAD (PRD),POST_DIAGNOSIS_SPREAD_LOC (PRD),LOCAL_RECURRENCE (PRD),DISEASE_FREE_NOW (PRD),SUPPORT_MEMBERSHIP (PRD),EHR_EXTRACTED,Age (Combined),PRIMARY SITE (Combined),RAAS_LAAS_Class,NUM MET SITES EVER (EHR_EXTRACTED)
0,RP-1447_ASCProject_0003_SALIVA_v1_Exome_OnPrem,ASCProject_0003_SALIVA,Normal,ASCProject_0003,DNA,None,ASCProject_0003_SALIVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ASCProject_0003,Breast (Parenchymal),No Mets,Breast (Parenchymal),Breast,No Mets,No Mets,NaN,5.0,38,F,BREAST (PARENCHYMAL),RIGHT BREAST,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,2057,BREAST (PARENCHYMAL),0.0,0.0,3,DIAGNOSED,38.800000,NaN,NaN,NaN,YES,YES,YES,AFTER,NaN,YES,YES,NaN,NaN,BREAST,NO,NaN,NO,NaN,NO,YES,YES,True,38.000000,BREAST (PARENCHYMAL),Non-RAAS/LAAS,0.0
1,RP-1447_ASCProject_0003_T1_v1_Exome_OnPrem,ASCProject_0003_T1,Tumor,ASCProject_0003,DNA,T1,ASCProject_0003_T1,NaN,NaN,13.0,BREAST,RESECTION,YES,NO,NO,LOW,ASCProject_0003,Breast (Parenchymal),No Mets,Breast (Parenchymal),Breast,No Mets,No Mets,NaN,5.0,38,F,BREAST (PARENCHYMAL),RIGHT BREAST,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,2057,BREAST (PARENCHYMAL),0.0,0.0,3,DIAGNOSED,38.800000,NaN,NaN,NaN,YES,YES,YES,AFTER,NaN,YES,YES,NaN,NaN,BREAST,NO,NaN,NO,NaN,NO,YES,YES,True,38.000000,BREAST (PARENCHYMAL),Non-RAAS/LAAS,0.0
2,RP-1447_ASCProject_0003_T3_v2_Exome_OnPrem,ASCProject_0003_T3,Tumor,ASCProject_0003,DNA,T3,ASCProject_0003_T3,NaN,NaN,26.0,BREAST,NaN,NaN,NaN,NaN,NaN,ASCProject_0003,Breast (Parenchymal),No Mets,Breast (Parenchymal),Breast,No Mets,No Mets,NaN,5.0,38,F,BREAST (PARENCHYMAL),RIGHT BREAST,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO METS,NO METS,2057,BREAST (PARENCHYMAL),0.0,0.0,3,DIAGNOSED,38.800000,NaN,NaN,NaN,YES,YES,YES,AFTER,NaN,YES,YES,NaN,NaN,BREAST,NO,NaN,NO,NaN,NO,YES,YES,True,38.000000,BREAST (PARENCHYMAL),Non-RAAS/LAAS,0.0
3,RP-1447_ASCProject_0005_BLOOD_Normal_v1_Exome_...,ASCProject_0005_BLOOD_Normal,Normal,ASCProject_0005,DNA,None,ASCProject_0005_BLOOD_Normal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ASCProject_0005,Breast (Parenchymal),"Trunk, Breast, Extremities, Bone",Breast (Parenchymal),Breast,Trunk,"Trunk, Breast, Extremities, Bone",NaN,5.0,31,F,BREAST (PARENCHYMAL),LEFT BREAST,0.0,NO,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3615,TRUNK,"TRUNK, OTHER (RIG

In [9]:
sample_patient_merged.to_csv(f"{PROJECT_DIR}/data/processed/sample_clin_data.tsv",sep="\t",index=False)